In [ ]:
import os

date = "mother_folder"
rnn_folder = f"base_path/{date}_DL"

shap_number = "project_name"
os.mkdir(f"{rnn_folder}/model_{shap_number}")

In [ ]:
fs = 20  # sampling rate
calc_start = 5
calc_end = 39

In [3]:
import numpy as np

signal1 = np.load("D:/Python_TK_3/datas/250514_GCaMP/250428-250514_ChangeOri_trace1/trace.npy")
signal2 = np.load("D:/Python_TK_3/datas/250522_GCaMP/250520-250522_Angle_trace1/trace.npy")
signal3 = np.load("D:/Python_TK_3/datas/250514_GCaMP/250428-250514_SlowFast_trace1/trace.npy")
signal4 = np.load("D:/Python_TK_3/datas/250522_GCaMP/250520-250522_Speed_trace1/trace.npy")
signal5 = np.load("D:/Python_TK_3/datas/250514_GCaMP/250428-250514_StopGo_trace1/trace.npy")
signal6 = np.load("D:/Python_TK_3/datas/250522_GCaMP/250520-250522_StopGo_trace1/trace.npy")

signal1 = np.concatenate((signal1, signal2), axis=1)
signal2 = np.concatenate((signal3, signal4), axis=1)
signal3 = np.concatenate((signal5, signal6), axis=1)

print(signal1.shape)
print(signal2.shape)
print(signal3.shape)

(26, 40800)
(26, 38760)
(26, 39440)


In [4]:
import numpy as np

pupil1 = np.load("D:/Python_TK_3/datas/250514_GCaMP/250428-250514_ChangeOri_pupil1/pupil_mm.npy")
pupil2 = np.load("D:/Python_TK_3/datas/250522_GCaMP/250520-250522_Angle_pupil1/pupil_mm.npy")
pupil3 = np.load("D:/Python_TK_3/datas/250514_GCaMP/250428-250514_SlowFast_pupil1/pupil_mm.npy")
pupil4 = np.load("D:/Python_TK_3/datas/250522_GCaMP/250520-250522_Speed_pupil1/pupil_mm.npy")
pupil5 = np.load("D:/Python_TK_3/datas/250514_GCaMP/250428-250514_StopGo_pupil1/pupil_mm.npy")
pupil6 = np.load("D:/Python_TK_3/datas/250522_GCaMP/250520-250522_StopGo_pupil1/pupil_mm.npy")

pupil1 = np.concatenate((pupil1[:, int(fs*calc_start):int(fs*calc_end)], pupil2[:, int(fs*calc_start):int(fs*calc_end)]), axis=0)
pupil2 = np.concatenate((pupil3[:, int(fs*calc_start):int(fs*calc_end)], pupil4[:, int(fs*calc_start):int(fs*calc_end)]), axis=0)
pupil3 = np.concatenate((pupil5[:, int(fs*calc_start):int(fs*calc_end)], pupil6[:, int(fs*calc_start):int(fs*calc_end)]), axis=0)

print(pupil1.shape)
print(pupil2.shape)
print(pupil3.shape)

(60, 680)
(57, 680)
(58, 680)


In [ ]:
first_ex = 0
last_ex = pupil1.shape[0] + pupil2.shape[0] + pupil3.shape[0]
#last_ex = 58
NumberOfDatas = last_ex - first_ex + 1        # number of experiments
start_stim = 20
stop_stim = 30
start_ave= 10
end_ave= 20
look_frame = 10               # read the previous and next n frames as input

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt

def signal_lowpass_filter(data):

    cutoff = 2  
    order = 4  
    nyquist = 0.5 * fs  
    normal_cutoff = cutoff / nyquist  


    b, a = butter(order, normal_cutoff, btype='low', analog=False)


    filtered_data = filtfilt(b, a, data)

    return filtered_data

In [ ]:
import numpy as np

def moving_average(data, window=int(fs*0.2)):
    weights = np.ones(window) / window
    return np.convolve(data, weights, mode='valid')

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt

def pupil_lowpass_filter(data):


    cutoff = 1
    order = 4  
    nyquist = 0.5 * fs  
    normal_cutoff = cutoff / nyquist  


    b, a = butter(order, normal_cutoff, btype='low', analog=False)

    filtered_data = filtfilt(b, a, data)

    return filtered_data

In [ ]:
import numpy as np

def z_score_calc(array1d):
    z_score = (array1d - np.mean(array1d)) / np.std(array1d)
    return z_score


def mini_max_norm(array_1d):

    min_val = array_1d.min()
    max_val = array_1d.max()

    if max_val - min_val == 0:
        arr_norm = np.zeros_like(array_1d)
    else:
        arr_norm = (array_1d - min_val) / (max_val - min_val)

    return arr_norm

def baseline_norm(array1d):
    #mean_array = np.mean(array1d[fs*(start_ave-calc_start):fs*(end_ave-calc_start)], axis=0)
    mean_array = np.mean(array1d[fs*start_ave:fs*end_ave], axis=0)
    temp_normalized = array1d / mean_array

    return temp_normalized

In [ ]:
import numpy as np


raw_signal = np.load("roi_trace_path")

raw_pupil  = np.concatenate((pupil1, pupil2, pupil3), axis=0)
#raw_signal = np.concatenate((signal1, signal2, signal3), axis=1)

#raw_pupil  = pupil1
#raw_signal = signal1

print(raw_pupil.shape)
print(raw_signal.shape)

(175, 680)
(23, 159120)


In [11]:
raw_signal = raw_signal[:, 59*680:]
print(raw_signal.shape)

(23, 119000)


In [12]:
pupil = raw_pupil
print(pupil.shape)

(175, 680)


In [13]:
pupil = pupil[first_ex:last_ex+1]
print(pupil.shape)

(175, 680)


In [14]:
#window = int(fs*0.4)

filtered_pupil = []
for ex in range(pupil.shape[0]):
    temp_filtered = pupil_lowpass_filter(pupil[ex])
    #temp_filtered = moving_average(pupil[ex], window=window)
    filtered_pupil.append(temp_filtered)

filtered_pupil = np.array(filtered_pupil)
print(filtered_pupil.shape)

(175, 680)


In [15]:
normalized_pupil = []
for ex in range(filtered_pupil.shape[0]):
    temp_normalized = baseline_norm(filtered_pupil[ex])
    normalized_pupil.append(temp_normalized)

normalized_pupil = np.array(normalized_pupil)
print(normalized_pupil.shape)

(175, 680)


In [16]:
pupil_z_score = []
for ex in range(filtered_pupil.shape[0]):
    pupil_z = z_score_calc(filtered_pupil[ex])
    pupil_z_score.append(pupil_z)

pupil_z_score = np.array(pupil_z_score)
print(pupil_z_score.shape)

(175, 680)


In [ ]:
start_value = 0.0

increment = 0.05

count = int(fs*(calc_end-calc_start))

t = [start_value + i * increment for i in range(count)]

In [18]:
dt = 1/fs

differential_pupil = normalized_pupil  # %

n_pupil_velocity = []
for ex in range(differential_pupil.shape[0]):
    temp_v = []
    for f in range(differential_pupil.shape[1]-1):
        temp_v.append((differential_pupil[ex, f+1]-differential_pupil[ex, f])/dt)
    n_pupil_velocity.append(temp_v)

n_pupil_velocity = np.array(n_pupil_velocity)
print(n_pupil_velocity.shape)

#n_pupil_velocity = n_pupil_velocity[:, int(fs*calc_start):int(fs*calc_end)]

(175, 679)


In [19]:
filtered_velocity = []
for ex in range(n_pupil_velocity.shape[0]):
    temp_v = pupil_lowpass_filter(n_pupil_velocity[ex])
    filtered_velocity.append(temp_v)

filtered_velocity = np.array(filtered_velocity)
print(filtered_velocity.shape)

(175, 679)


In [20]:
learn_start = start_stim - calc_start - 1.0
learn_end   = start_stim - calc_start + 9
frames = int(fs*(learn_end-learn_start))

In [21]:
extracted_pupil = n_pupil_velocity[:, int(fs*learn_start)-look_frame:int(fs*learn_start)+frames]

binary_pupil = []
for ex in range(extracted_pupil.shape[0]):
    temp_p = []
    for f in range(extracted_pupil.shape[1]):
        if extracted_pupil[ex, f] > 0.10:
            temp_p.append(1)
        else:
            temp_p.append(0)
    binary_pupil.append(temp_p)

binary_pupil = np.array(binary_pupil)
print(binary_pupil.shape)

(175, 210)


In [22]:
#Pupil_data = filtered_pupil[:, fs*calc_start+3:fs*calc_end]
Pupil_data = binary_pupil

In [23]:
ca_array = np.transpose(raw_signal)

print(raw_signal.shape)
print(ca_array.shape)

(23, 119000)
(119000, 23)


In [24]:
#ca_array = ca_array[:, :27]

In [ ]:
import numpy as np

batch_size = fs*(calc_end-calc_start)
pre_image_array = [ca_array[i:i+batch_size] for i in range(0, ca_array.shape[0], batch_size)]

pre_image_array = np.array(pre_image_array)
print(pre_image_array.shape)

(175, 680, 23)


In [26]:
#learn_start = start_stim - calc_start - 1.0
#learn_end   = 34
#frames = int(fs*(learn_end-learn_start))

In [27]:
window = int(fs*0.2)

image_array = []
for ex in range (pre_image_array.shape[0]):
    #image_array.append(pre_image_array[ex, fs*calc_start:fs*calc_end])
    #image_array.append(pre_image_array[ex, int(fs*learn_start)-look_frame-window+1:int(fs*learn_start)+frames])
    image_array.append(pre_image_array[ex])

image_array = np.array(image_array)

print(image_array.shape)

(175, 680, 23)


In [28]:
filtered_ca = []
for ex in range(image_array.shape[0]):
    x = []
    for i in range(image_array.shape[2]):
        raw_ratio = [row[i] for row in image_array[ex]]
        filtered_array = signal_lowpass_filter(np.array(raw_ratio))
        #filtered_array = moving_average(np.array(raw_ratio), window=window)
        filtered_ratio = filtered_array.tolist()
        x.append(filtered_ratio)
    filtered_ca.append(x)

filtered_ca = np.array(filtered_ca)
filtered_ca = filtered_ca.transpose(0, 2, 1)

print(filtered_ca.shape)

(175, 680, 23)


In [29]:
filtered_ca = filtered_ca[:, 3:]
print(filtered_ca.shape)

(175, 677, 23)


In [30]:
signal_z = []

for ex in range(filtered_ca.shape[0]):
    temp_signal = []
    for area in range (filtered_ca.shape[2]):
        temp_area = z_score_calc(filtered_ca[ex, :, area])
        temp_signal.append(temp_area)
    signal_z.append(temp_signal)

signal_z = np.array(signal_z)

print(signal_z.shape)

(175, 23, 677)


In [31]:
signal = filtered_ca.transpose(0, 2, 1)
print(signal.shape)
#signal = signal[:, :, :-1]
#print(signal.shape)

(175, 23, 677)


In [ ]:
roi_trace = np.load("roi_trace_path")
print(roi_trace.shape)

(35, 159120)


In [33]:
roi_trace = roi_trace[:, 59*680:]
print(roi_trace.shape)

(35, 119000)


In [34]:
region_num = 27
# window = int(fs*0.2)

binary_pupil = []
extracted_signal = []
extracted_z_area = []
max_idxs = []
for ex in range(filtered_velocity.shape[0]):
    max_idx = np.argmax(filtered_velocity[ex, int(fs*(start_stim-calc_start)):int(fs*(start_stim-calc_start+9))]) + int(fs*(start_stim-calc_start))
    extracted_pupil = filtered_velocity[ex, max_idx-int(fs*0.65)-look_frame:max_idx+int(fs*0.05)]
    extracted_ca = filtered_ca[ex, max_idx-int(fs*0.65)-look_frame-window+2:max_idx+int(fs*0.05)-window+2]
    roi_z = z_score_calc(roi_trace[region_num, ex*680:(ex+1)*680])
    #extracted_z = signal_z[ex, region_num, max_idx-int(fs*0.65)-look_frame-window+2:max_idx+int(fs*0.05)-window+2]
    extracted_z = roi_z[max_idx-int(fs*0.65)-look_frame-window+2:max_idx+int(fs*0.05)-window+2]
    temp_p = []
    for f in range(extracted_pupil.shape[0]):
        if extracted_pupil[f] > 0.1:
            temp_p.append(1)
        else:
            temp_p.append(0)
    binary_pupil.append(temp_p)
    max_idxs.append(max_idx)
    extracted_signal.append(extracted_ca)
    extracted_z_area.append(extracted_z)

binary_pupil = np.array(binary_pupil)
extracted_signal = np.array(extracted_signal)
extracted_z_area = np.array(extracted_z_area)
print(binary_pupil.shape)
print(extracted_signal.shape, extracted_z_area.shape)

(175, 24)
(175, 24, 23) (175, 24)


In [35]:
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_pre_pupil.npy", binary_pupil)

In [36]:
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_peak_frame.npy", np.array(max_idxs))

In [37]:
Pupil_data = binary_pupil
signal = extracted_signal

target = []
feature = []
experiment_list = []
for ex in range(Pupil_data.shape[0]):
    if np.all(Pupil_data[ex, :int(fs*0.65)] == 0) and np.any(Pupil_data[ex, int(fs*0.65):] == 1) and np.any(extracted_z_area[ex, int(fs*0.65):int(fs*1.1)] > 1.8):
        target.append(Pupil_data[ex])
        feature.append(signal[ex])
        experiment_list.append(ex)

Pupil_data = np.array(target)
signal = np.array(feature).transpose(0, 2, 1)
experiment_list = np.array(experiment_list)

print(Pupil_data.shape)
print(signal.shape)
print(experiment_list.shape)

(26, 24)
(26, 23, 24)
(26,)


In [38]:
Pupil_data2 = binary_pupil
signal2 = extracted_signal

target2 = []
feature2 = []
experiment_list2 = []
for ex in range(Pupil_data2.shape[0]):
    if np.all(Pupil_data2[ex, :int(fs*0.65)] == 0) and np.all(Pupil_data2[ex, int(fs*0.65):] == 0) and np.all(extracted_z_area[ex, int(fs*0.65):int(fs*1.1)] < 1.2):
        target2.append(Pupil_data2[ex])
        feature2.append(signal2[ex])
        experiment_list2.append(ex)

Pupil_data2 = np.array(target2)
signal2 = np.array(feature2).transpose(0, 2, 1)
experiment_list2 = np.array(experiment_list2)

print(Pupil_data2.shape)
print(signal2.shape)
print(experiment_list2.shape)

(26, 24)
(26, 23, 24)
(26,)


In [42]:
import random
random.seed(123)

#random_success = random.sample(experiment_list.tolist(), 20)
#random_success = np.array(random_success)
ex_list_mix = np.concatenate((experiment_list, experiment_list2), axis=0)

print(experiment_list.shape, ex_list_mix.shape)

(26,) (52,)


In [43]:
print(experiment_list)
print(experiment_list2)

[  2  22  25  35  43  47  49  54  60  73  91 114 117 121 127 129 133 143
 146 150 152 159 162 165 169 171]
[  4  10  11  13  19  24  37  42  46  57  67  69  74  77  80  83  93  94
 101 103 107 118 123 132 135 157]


In [44]:
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_selected_experiments_success.npy", experiment_list)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_selected_experiments_failure.npy", experiment_list2)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_selected_experiments.npy", ex_list_mix)

In [45]:
Pupil_data2 = binary_pupil
signal2 = extracted_signal

target2 = []
feature2 = []
experiment_list2 = []
for ex in range(Pupil_data2.shape[0]):
    if np.all(Pupil_data2[ex, :int(fs*0.65)] == 0) and np.all(Pupil_data2[ex, int(fs*0.65):] == 0) and np.any(extracted_z_area[ex, int(fs*0.65):int(fs*1.1)] > 1.75):
        target2.append(Pupil_data2[ex])
        feature2.append(signal2[ex])
        experiment_list2.append(ex)

Pupil_data2 = np.array(target2)
signal2 = np.array(feature2).transpose(0, 2, 1)
experiment_list2 = np.array(experiment_list2)

print(Pupil_data2.shape)
print(signal2.shape)
print(experiment_list2.shape)

(13, 24)
(13, 23, 24)
(13,)


In [40]:
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_selected_experiments_excluded.npy", experiment_list)